# Bribri TTS: Spark-TTS Fine-Tuning

COSC 72 Final Project — Spring 2026
  
Ikenna Nwafor & Zachary Shah

Ikenna.27@dartmouth.edu

Zachary.s.shah.27@dartmouth.edu

Fine-tunes [Spark-TTS-0.5B](https://huggingface.co/SparkAudio/Spark-TTS-0.5B) on ~25 minutes of Bribri audio (449 train / 56 valid / 56 test clips). Spark-TTS is an LLM-based TTS system built on Qwen2.5-0.5B with a separate audio codec (BiCodec), released March 2025. To our knowledge, this is among the first applications of Spark-TTS to an indigenous language of the Americas.

## Notebook source

Adapted from [Unsloth's Spark-TTS 0.5B fine-tuning template](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Spark_TTS_(0_5B).ipynb). The Unsloth notebook's defaults assume English target speech; we replace the dataset loading with the Bribri splits and the inference text with our test sentences, but otherwise leave the training recipe unchanged.

## Runtime

Use a Colab **L4 GPU** as the notebook is configured for bf16 LoRA training, so a T4 will not work without modifications.

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!git clone https://github.com/SparkAudio/Spark-TTS
!pip install omegaconf einx torchcodec "datasets>=3.4.1,<4.0.0"

### Unsloth


In [ ]:
from unsloth import FastModel
import torch
from huggingface_hub import snapshot_download

max_seq_length = 2048 # Choose any for long context!

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",
    # Qwen3 new models
    "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    # Other very popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

# Download model and code
snapshot_download("unsloth/Spark-TTS-0.5B", local_dir = "Spark-TTS-0.5B")

model, tokenizer = FastModel.from_pretrained(
    model_name = f"Spark-TTS-0.5B/LLM",
    max_seq_length = max_seq_length,
    dtype = torch.float32, # Spark seems to only work on float32 for now
    full_finetuning = True, # We support full finetuning now!
    load_in_4bit = False,
    #token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


.gitattributes: 0.00B [00:00, ?B/s]

config.yaml: 0.00B [00:00, ?B/s]

BiCodec/model.safetensors:   0%|          | 0.00/626M [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

wav2vec2-large-xlsr-53/pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

==((====))==  Unsloth 2026.5.8: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Float16 full finetuning uses more memory since we upcast weights to float32.


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
#LoRA does not work with float32 only works with bfloat16 !
model = FastModel.get_peft_model(
    model,
    r = 128, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 128,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Full finetuning is enabled, so .get_peft_model has no effect


<a name="Data"></a>
### Data Prep

We load the Bribri training split (449 clips) from our project CSV + WAV folders on Drive. The dataset is built as a HuggingFace `Dataset` with two columns: `text` (NFD-normalized Bribri sentences) and `audio` (16 kHz mono clips). This matches Spark-TTS's expected single-speaker format. The CSV (`bribri-wav2vec2-train.csv`) provides the transcripts and audio paths; we rewrite the paths to point at our mounted Drive shortcut and apply Unicode NFD normalization so Bribri's tone diacritics become separate combining-mark codepoints. The downstream tokenizer expects `text` as the field name; the schema is otherwise identical to the Elise English-language dataset that the Unsloth template uses by default.

In [ ]:
import pandas as pd, os, unicodedata
from datasets import Dataset, Audio
from google.colab import drive
drive.mount('/content/drive')

CSV_DIR = '/content/drive/MyDrive/Junior/cosc72/csv'
WAV_DIR = '/content/drive/MyDrive/Junior/cosc72/wav'

train_df = pd.read_csv(f'{CSV_DIR}/bribri-wav2vec2-train.csv')
train_df['path'] = train_df['path'].apply(lambda p: f'{WAV_DIR}/{os.path.basename(p)}')
train_df['text'] = train_df['sentence'].apply(lambda s: unicodedata.normalize('NFD', s).replace('0',''))

dataset = Dataset.from_dict({
    'audio': train_df['path'].tolist(),
    'text':  train_df['text'].tolist(),
}).cast_column('audio', Audio(sampling_rate=16000))

print(dataset)
print(dataset[0])

Mounted at /content/drive
Dataset({
    features: ['audio', 'text'],
    num_rows: 449
})
{'audio': {'path': '/content/drive/MyDrive/Junior/cosc72/wav/JL-dialogue-139.wav', 'array': array([ 7.32421875e-04,  2.13623047e-04, -4.57763672e-04, ...,
        3.05175781e-05,  6.10351562e-05,  5.49316406e-04]), 'sampling_rate': 16000}, 'text': "kèq i kiànẽ tkèĩã sulèq wa éxmã se' dòqrànẽ se' icha wa shkãmẽ'la wa"}


In [ ]:
#@title Tokenization Function

import locale
import torchaudio.transforms as T
import os
import torch
import sys
import numpy as np
sys.path.append('Spark-TTS')
from sparktts.models.audio_tokenizer import BiCodecTokenizer
from sparktts.utils.audio import audio_volume_normalize

audio_tokenizer = BiCodecTokenizer("Spark-TTS-0.5B", "cuda")
def extract_wav2vec2_features( wavs: torch.Tensor) -> torch.Tensor:
        """extract wav2vec2 features"""

        if wavs.shape[0] != 1:

             raise ValueError(f"Expected batch size 1, but got shape {wavs.shape}")
        wav_np = wavs.squeeze(0).cpu().numpy()

        processed = audio_tokenizer.processor(
            wav_np,
            sampling_rate = 16000,
            return_tensors = "pt",
            padding = True,
        )
        input_values = processed.input_values

        input_values = input_values.to(audio_tokenizer.feature_extractor.device)

        model_output = audio_tokenizer.feature_extractor(
            input_values,
        )


        if model_output.hidden_states is None:
             raise ValueError("Wav2Vec2Model did not return hidden states. Ensure config `output_hidden_states=True`.")

        num_layers = len(model_output.hidden_states)
        required_layers = [11, 14, 16]
        if any(l >= num_layers for l in required_layers):
             raise IndexError(f"Requested hidden state indices {required_layers} out of range for model with {num_layers} layers.")

        feats_mix = (
            model_output.hidden_states[11] + model_output.hidden_states[14] + model_output.hidden_states[16]
        ) / 3

        return feats_mix
def formatting_audio_func(example):
    text = f"{example['source']}: {example['text']}" if "source" in example else example["text"]
    audio_array = example["audio"]["array"]
    sampling_rate = example["audio"]["sampling_rate"]

    target_sr = audio_tokenizer.config['sample_rate']

    if sampling_rate != target_sr:
        resampler = T.Resample(orig_freq = sampling_rate, new_freq = target_sr)
        audio_tensor_temp = torch.from_numpy(audio_array).float()
        audio_array = resampler(audio_tensor_temp).numpy()

    if audio_tokenizer.config["volume_normalize"]:
        audio_array = audio_volume_normalize(audio_array)

    ref_wav_np = audio_tokenizer.get_ref_clip(audio_array)

    audio_tensor = torch.from_numpy(audio_array).unsqueeze(0).float().to(audio_tokenizer.device)
    ref_wav_tensor = torch.from_numpy(ref_wav_np).unsqueeze(0).float().to(audio_tokenizer.device)


    feat = extract_wav2vec2_features(audio_tensor)

    batch = {

        "wav": audio_tensor,
        "ref_wav": ref_wav_tensor,
        "feat": feat.to(audio_tokenizer.device),
    }


    semantic_token_ids, global_token_ids = audio_tokenizer.model.tokenize(batch)

    global_tokens = "".join(
        [f"<|bicodec_global_{i}|>" for i in global_token_ids.squeeze().cpu().numpy()] # Squeeze batch dim
    )
    semantic_tokens = "".join(
        [f"<|bicodec_semantic_{i}|>" for i in semantic_token_ids.squeeze().cpu().numpy()] # Squeeze batch dim
    )

    inputs = [
        "<|task_tts|>",
        "<|start_content|>",
        text,
        "<|end_content|>",
        "<|start_global_token|>",
        global_tokens,
        "<|end_global_token|>",
        "<|start_semantic_token|>",
        semantic_tokens,
        "<|end_semantic_token|>",
        "<|im_end|>"
    ]
    inputs = "".join(inputs)
    return {"text": inputs}


dataset = dataset.map(formatting_audio_func, remove_columns = ["audio"])
print("Moving Bicodec model and Wav2Vec2Model to cpu.")
audio_tokenizer.model.cpu()
audio_tokenizer.feature_extractor.cpu()
torch.cuda.empty_cache()

Missing tensor: mel_transformer.spectrogram.window
Missing tensor: mel_transformer.mel_scale.fb


Parameter 'function'=<function formatting_audio_func at 0x7ade67f63420> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/449 [00:00<?, ? examples/s]

Moving Bicodec model and Wav2Vec2Model to cpu.


<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!!

In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = False, # We're doing full float32 so disable mixed precision
        bf16 = True, # We're doing full float32 so disable mixed precision
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/449 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA L4. Max memory = 22.034 GB.
5.029 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 449 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 506,634,112 of 506,634,112 (100.00% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,9.688400
2,9.405000
3,9.451700
4,8.998600
5,8.687900
6,7.783300
7,7.737000
8,7.514700
9,7.359300
10,7.207900


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

87.8713 seconds used for training.
1.46 minutes used for training.
Peak reserved memory = 7.096 GB.
Peak reserved memory for training = 2.067 GB.
Peak reserved memory % of max memory = 32.205 %.
Peak reserved memory for training % of max memory = 9.381 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the prompts

In [ ]:
input_text = "ìs be' shkẽ̀nã"
chosen_voice = None

In [ ]:
#@title Run Inference

import torch
import re
import numpy as np
from typing import Dict, Any
import torchaudio.transforms as T

FastModel.for_inference(model) # Enable native 2x faster inference

@torch.inference_mode()
def generate_speech_from_text(
    text: str,
    temperature: float = 0.8,   # Generation temperature
    top_k: int = 50,            # Generation top_k
    top_p: float = 1,        # Generation top_p
    max_new_audio_tokens: int = 2048, # Max tokens for audio part
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
) -> np.ndarray:
    """
    Generates speech audio from text using default voice control parameters.

    Args:
        text (str): The text input to be converted to speech.
        temperature (float): Sampling temperature for generation.
        top_k (int): Top-k sampling parameter.
        top_p (float): Top-p (nucleus) sampling parameter.
        max_new_audio_tokens (int): Max number of new tokens to generate (limits audio length).
        device (torch.device): Device to run inference on.

    Returns:
        np.ndarray: Generated waveform as a NumPy array.
    """

    torch.compiler.reset()

    prompt = "".join([
        "<|task_tts|>",
        "<|start_content|>",
        text,
        "<|end_content|>",
        "<|start_global_token|>"
    ])

    model_inputs = tokenizer([prompt], return_tensors = "pt").to(device)

    print("Generating token sequence...")
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens = max_new_audio_tokens, # Limit generation length
        do_sample = True,
        temperature = temperature,
        top_k = top_k,
        top_p = top_p,
        eos_token_id = tokenizer.eos_token_id, # Stop token
        pad_token_id = tokenizer.pad_token_id # Use models pad token id
    )
    print("Token sequence generated.")


    generated_ids_trimmed = generated_ids[:, model_inputs.input_ids.shape[1]:]


    predicts_text = tokenizer.batch_decode(generated_ids_trimmed, skip_special_tokens = False)[0]
    # print(f"\nGenerated Text (for parsing):\n{predicts_text}\n") # Debugging

    # Extract semantic token IDs using regex
    semantic_matches = re.findall(r"<\|bicodec_semantic_(\d+)\|>", predicts_text)
    if not semantic_matches:
        print("Warning: No semantic tokens found in the generated output.")
        # Handle appropriately - perhaps return silence or raise error
        return np.array([], dtype = np.float32)

    pred_semantic_ids = torch.tensor([int(token) for token in semantic_matches]).long().unsqueeze(0) # Add batch dim

    # Extract global token IDs using regex (assuming controllable mode also generates these)
    global_matches = re.findall(r"<\|bicodec_global_(\d+)\|>", predicts_text)
    if not global_matches:
         print("Warning: No global tokens found in the generated output (controllable mode). Might use defaults or fail.")
         pred_global_ids = torch.zeros((1, 1), dtype = torch.long)
    else:
         pred_global_ids = torch.tensor([int(token) for token in global_matches]).long().unsqueeze(0) # Add batch dim

    pred_global_ids = pred_global_ids.unsqueeze(0) # Shape becomes (1, 1, N_global)

    print(f"Found {pred_semantic_ids.shape[1]} semantic tokens.")
    print(f"Found {pred_global_ids.shape[2]} global tokens.")


    # 5. Detokenize using BiCodecTokenizer
    print("Detokenizing audio tokens...")
    # Ensure audio_tokenizer and its internal model are on the correct device
    audio_tokenizer.device = device
    audio_tokenizer.model.to(device)
    # Squeeze the extra dimension from global tokens as seen in SparkTTS example
    wav_np = audio_tokenizer.detokenize(
        pred_global_ids.to(device).squeeze(0), # Shape (1, N_global)
        pred_semantic_ids.to(device)           # Shape (1, N_semantic)
    )
    print("Detokenization complete.")

    return wav_np

if __name__ == "__main__":
    print(f"Generating speech for: '{input_text}'")
    text = f"{chosen_voice}: " + input_text if chosen_voice else input_text
    generated_waveform = generate_speech_from_text(input_text)

    if generated_waveform.size > 0:
        import soundfile as sf
        output_filename = "generated_speech_controllable.wav"
        sample_rate = audio_tokenizer.config.get("sample_rate", 16000)
        sf.write(output_filename, generated_waveform, sample_rate)
        print(f"Audio saved to {output_filename}")

        # Optional: Play in notebook
        from IPython.display import Audio, display
        display(Audio(generated_waveform, rate = sample_rate))
    else:
        print("Audio generation failed (no tokens found?).")

Generating speech for: 'ìs be' shkẽ̀nã'
Generating token sequence...
Token sequence generated.
Found 74 semantic tokens.
Found 32 global tokens.
Detokenizing audio tokens...
Detokenization complete.
Audio saved to generated_speech_controllable.wav


### Generate 20 fixed test samples for human evaluation

Synthesize the same 20 test sentences used in the cjp and spa conditions (deterministic selection: NFD-length sort, stride sampling) so listeners can A/B compare per-sentence across all three models. Saves WAVs to `outputs/sparktts/samples_for_eval/sample_NN.wav` on Drive.

In [ ]:
import pandas as pd, unicodedata, os, soundfile as sf

# Same 20-sentence selection as cjp/spa
test_df = pd.read_csv(f'{CSV_DIR}/bribri-wav2vec2-test.csv')
test_df['sentence_nfd'] = test_df['sentence'].apply(lambda s: unicodedata.normalize('NFD', s).replace('0',''))
test_df['len'] = test_df['sentence_nfd'].str.len()
sorted_df = test_df.sort_values('len').reset_index(drop=True)
stride = max(1, len(sorted_df) // 20)
chosen = sorted_df.iloc[::stride].head(20).reset_index(drop=True)

SAMPLES_DIR = '/content/drive/MyDrive/Junior/cosc72/outputs/sparktts/samples_for_eval'
os.makedirs(SAMPLES_DIR, exist_ok=True)

sample_rate = audio_tokenizer.config.get("sample_rate", 16000)
for i, row in chosen.iterrows():
    wav = generate_speech_from_text(row['sentence'])
    if wav.size > 0:
        sf.write(f'{SAMPLES_DIR}/sample_{i:02d}.wav', wav, sample_rate)
        print(f"  {i:02d}: ok ({wav.shape[0]/sample_rate:.1f}s)")
    else:
        print(f"  {i:02d}: FAILED (no tokens)")
print("Done.")

Generating token sequence...
Token sequence generated.
Found 41 semantic tokens.
Found 32 global tokens.
Detokenizing audio tokens...
Detokenization complete.
  00: ok (0.8s)
Generating token sequence...
Token sequence generated.
Found 52 semantic tokens.
Found 32 global tokens.
Detokenizing audio tokens...
Detokenization complete.
  01: ok (1.0s)
Generating token sequence...
Token sequence generated.
Found 62 semantic tokens.
Found 32 global tokens.
Detokenizing audio tokens...
Detokenization complete.
  02: ok (1.2s)
Generating token sequence...
Token sequence generated.
Found 47 semantic tokens.
Found 32 global tokens.
Detokenizing audio tokens...
Detokenization complete.
  03: ok (0.9s)
Generating token sequence...
Token sequence generated.
Found 56 semantic tokens.
Found 32 global tokens.
Detokenizing audio tokens...
Detokenization complete.
  04: ok (1.1s)
Generating token sequence...
Token sequence generated.
Found 67 semantic tokens.
Found 32 global tokens.
Detokenizing audio t

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/Junior/cosc72/outputs/sparktts/model'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

('/content/drive/MyDrive/Junior/cosc72/outputs/sparktts/model/tokenizer_config.json',
 '/content/drive/MyDrive/Junior/cosc72/outputs/sparktts/model/special_tokens_map.json',
 '/content/drive/MyDrive/Junior/cosc72/outputs/sparktts/model/chat_template.jinja',
 '/content/drive/MyDrive/Junior/cosc72/outputs/sparktts/model/vocab.json',
 '/content/drive/MyDrive/Junior/cosc72/outputs/sparktts/model/merges.txt',
 '/content/drive/MyDrive/Junior/cosc72/outputs/sparktts/model/added_tokens.json',
 '/content/drive/MyDrive/Junior/cosc72/outputs/sparktts/model/tokenizer.json')

In [ ]:
for i, row in chosen.iterrows():
    print(f"{i:02d} ({row['len']:3d}ch): {row['sentence']!r}")

00 (  2ch): 'no'
01 (  6ch): 'ũjùx'
02 (  7ch): 'úxjùx'
03 (  9ch): 'ùxùxùx'
04 ( 17ch): "e's ie i wà dör"
05 ( 19ch): "e'tã ie'pa ák ák"
06 ( 22ch): "se' wake' dawòqwãdà"
07 ( 23ch): 'tsîxnẽt tsîxnẽt èq'
08 ( 23ch): 'shléqwöla shléqwöla'
09 ( 24ch): "wèxs sĩnĩ' kũèx sö"
10 ( 28ch): "kali'wèx dànẽ sö sku' wa"
11 ( 30ch): "e'r ãwíxẽ tsìxnẽt kúkúr"
12 ( 33ch): 'kèq kapòqkũ i tàyë tã yéqs'
13 ( 35ch): "ajá e' kũéxkĩ kèq ie'pa i kuke"
14 ( 38ch): "îxñẽ e'tã ye' kèq míx a káxmîx"
15 ( 39ch): "ie'pa mĩkalè tã ikuòq i tòq tèkẽ"
16 ( 45ch): "e' tã káx wéx àxtã e' kĩ íyi tsikìnã"
17 ( 50ch): "e' mĩ'kẽ ie'pa ièkẽ ie'pa bolsala ã bolsa èq"
18 ( 54ch): "e' ã ie'r skà ie' dawè tso' e' kũéxkĩ e' ñèxke"
19 ( 58ch): "sa'' i' ël sa' éqltë tã wake' i bitséitsèkẽ kéxkra"


### Saving to float16

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("spark_tts_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/spark_tts_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("spark_tts_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/spark_tts_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("spark_tts_lora")
    tokenizer.save_pretrained("spark_tts_lora")
if False:
    model.push_to_hub("HF_USERNAME/spark_tts_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/spark_tts_lora", token = "YOUR_HF_TOKEN")

Unsloth: You have 1 CPUs. Using `safe_serialization` is 10x slower.
We shall switch to Pytorch saving, which might take 3 minutes and not 30 minutes.
To force `safe_serialization`, set it to `None` instead.
Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 15.1G

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 3.99 out of 12.67 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...

100%|██████████| 28/28 [00:01<00:00, 27.83it/s]

Unsloth: Saving tokenizer... Done.
Unsloth: Saving model/pytorch_model-00001-of-00002.bin...
Unsloth: Saving model/pytorch_model-00002-of-00002.bin...
Done.